# Step 2: Data Analysis and Anomaly Flagging

This notebook:
1. Profiles the clean data (regional stats, specialty distribution)
2. Adds anomaly detection flags to each facility
3. Identifies medical deserts (underserved regions)
4. Saves enriched data with flags to Unity Catalog Delta tables

**Run notebooks 00, 01 first!**

## 2A. Load Clean Data

In [0]:
import json
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, BooleanType, StringType

# ============================================================
# CONFIG: Must match what was set in 00_setup
# ============================================================
CATALOG = "hackathon_vf"
SCHEMA = "healthcare"
TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"

try:
    spark.sql(f"USE CATALOG {CATALOG}")
    spark.sql(f"USE SCHEMA {SCHEMA}")
except Exception:
    CATALOG = "hive_metastore"
    SCHEMA = "hackathon"
    TABLE_PREFIX = f"{CATALOG}.{SCHEMA}"
    spark.sql(f"USE {SCHEMA}")

df = spark.table(f"{TABLE_PREFIX}.facilities_clean")

# Convert string 'null' to actual nulls and cast to integer
df = df.withColumn("numberDoctors", F.when(F.col("numberDoctors") == "null", None).otherwise(F.col("numberDoctors").cast("int")))
df = df.withColumn("capacity", F.when(F.col("capacity") == "null", None).otherwise(F.col("capacity").cast("int")))

total = df.count()
print(f"Loaded {total} clean facilities from {TABLE_PREFIX}.facilities_clean")

Loaded 797 clean facilities from hackathon_vf.healthcare.facilities_clean


## 2B. Regional Statistics

In [0]:
print("=== FACILITIES PER REGION ===")
region_stats = df.groupBy("address_stateOrRegion").agg(
    F.count("*").alias("facility_count"),
    F.sum(F.when(F.col("numberDoctors").isNotNull(), F.col("numberDoctors")).otherwise(0)).alias("total_doctors"),
    F.sum(F.when(F.col("capacity").isNotNull(), F.col("capacity")).otherwise(0)).alias("total_beds"),
    F.countDistinct("facilityTypeId").alias("facility_types"),
).orderBy("facility_count", ascending=True)
region_stats.show(20, truncate=False)


=== FACILITIES PER REGION ===
+------------------------------------------+--------------+-------------+----------+--------------+
|address_stateOrRegion                     |facility_count|total_doctors|total_beds|facility_types|
+------------------------------------------+--------------+-------------+----------+--------------+
|ASHANTI REGION                            |1             |0            |0         |1             |
|Bono                                      |1             |0            |238       |1             |
|Techiman Municipal                        |1             |0            |0         |1             |
|Ghana                                     |1             |0            |0         |1             |
|Asokwa-Kumasi                             |1             |0            |0         |1             |
|Takoradi                                  |1             |0            |0         |1             |
|SH                                        |1             |0          

In [0]:
# UDFs for specialty analysis
@F.udf(IntegerType())
def count_json_items(json_str):
    if not json_str or json_str in ("null", "[]"):
        return 0
    try:
        items = json.loads(json_str)
        return len(items) if isinstance(items, list) else 0
    except (json.JSONDecodeError, TypeError):
        return 0

# Specialty coverage by region
df_specs = df.withColumn("specialty_count", count_json_items(F.col("specialties")))
print("=== SPECIALTY COVERAGE BY REGION ===")
df_specs.groupBy("address_stateOrRegion").agg(
    F.sum("specialty_count").alias("total_specialty_slots"),
    F.avg("specialty_count").alias("avg_specialties_per_facility"),
).orderBy("total_specialty_slots", ascending=True).show(20, truncate=False)

=== SPECIALTY COVERAGE BY REGION ===
+----------------------+---------------------+----------------------------+
|address_stateOrRegion |total_specialty_slots|avg_specialties_per_facility|
+----------------------+---------------------+----------------------------+
|Asutifi South         |1                    |1.0                         |
|Oti                   |1                    |1.0                         |
|Central Tongu District|1                    |1.0                         |
|Ejisu Municipal       |1                    |1.0                         |
|Ahafo                 |1                    |1.0                         |
|Upper West Region     |1                    |1.0                         |
|Sissala West District |1                    |1.0                         |
|Ledzokuku-Krowor      |1                    |1.0                         |
|Ahafo Ano South-East  |1                    |1.0                         |
|Upper East            |1                    |1.0  

In [0]:
print("=== FACILITY TYPE DISTRIBUTION ===")
df.groupBy("facilityTypeId").agg(
    F.count("*").alias("count"),
    F.avg("numberDoctors").alias("avg_doctors"),
    F.avg("capacity").alias("avg_beds"),
).show(10, truncate=False)

print("=== PUBLIC VS PRIVATE ===")
df.groupBy("operatorTypeId").count().show()

=== FACILITY TYPE DISTRIBUTION ===
+--------------+-----+-----------+--------+
|facilityTypeId|count|avg_doctors|avg_beds|
+--------------+-----+-----------+--------+
|null          |222  |NULL       |NULL    |
|clinic        |208  |NULL       |NULL    |
|hospital      |348  |12.5       |126.8   |
|dentist       |13   |NULL       |NULL    |
|farmacy       |5    |NULL       |NULL    |
|doctor        |1    |NULL       |NULL    |
+--------------+-----+-----------+--------+

=== PUBLIC VS PRIVATE ===
+--------------+-----+
|operatorTypeId|count|
+--------------+-----+
|          null|  619|
|       private|  125|
|        public|   53|
+--------------+-----+



## 2C. Anomaly Detection Flags

In [0]:
# Add count columns
df_enriched = (
    df
    .withColumn("num_procedures", count_json_items(F.col("procedure")))
    .withColumn("num_equipment", count_json_items(F.col("equipment")))
    .withColumn("num_capabilities", count_json_items(F.col("capability")))
    .withColumn("num_specialties", count_json_items(F.col("specialties")))
)

# FLAG 1: Many procedures but no/few doctors
df_enriched = df_enriched.withColumn("flag_procedures_no_doctors",
    F.when((F.col("num_procedures") > 5) &
           ((F.col("numberDoctors").isNull()) | (F.col("numberDoctors") < 2)), True
    ).otherwise(False))

# FLAG 2: Large capacity but no equipment listed
df_enriched = df_enriched.withColumn("flag_capacity_no_equipment",
    F.when((F.col("capacity").isNotNull()) & (F.col("capacity") > 50) &
           (F.col("num_equipment") == 0), True
    ).otherwise(False))

# FLAG 3: Clinic claims surgery specialty
df_enriched = df_enriched.withColumn("flag_clinic_claims_surgery",
    F.when((F.col("facilityTypeId") == "clinic") &
           (F.lower(F.col("specialties")).contains("surgery")), True
    ).otherwise(False))

# FLAG 4: Too many specialties for a small facility
df_enriched = df_enriched.withColumn("flag_too_many_specialties",
    F.when((F.col("num_specialties") > 5) &
           ((F.col("numberDoctors").isNull()) | (F.col("numberDoctors") < 5)), True
    ).otherwise(False))

# FLAG 5: Sparse record (no data at all)
df_enriched = df_enriched.withColumn("flag_sparse_record",
    F.when((F.col("num_procedures") == 0) & (F.col("num_equipment") == 0) &
           (F.col("num_capabilities") == 0) & (F.col("description").isNull()), True
    ).otherwise(False))

In [0]:
print("=== ANOMALY FLAG SUMMARY ===")
for flag_col in [c for c in df_enriched.columns if c.startswith("flag_")]:
    count = df_enriched.filter(F.col(flag_col) == True).count()
    print(f"  {flag_col}: {count} facilities flagged")

print("\n=== FACILITIES WITH PROCEDURE/DOCTOR MISMATCH ===")
df_enriched.filter(F.col("flag_procedures_no_doctors") == True).select(
    "name", "address_stateOrRegion", "facilityTypeId",
    "numberDoctors", "num_procedures", "num_equipment"
).show(10, truncate=False)

=== ANOMALY FLAG SUMMARY ===
  flag_procedures_no_doctors: 55 facilities flagged
  flag_capacity_no_equipment: 10 facilities flagged
  flag_clinic_claims_surgery: 19 facilities flagged
  flag_too_many_specialties: 94 facilities flagged
  flag_sparse_record: 0 facilities flagged

=== FACILITIES WITH PROCEDURE/DOCTOR MISMATCH ===
+---------------------------------------+---------------------+--------------+-------------+--------------+-------------+
|name                                   |address_stateOrRegion|facilityTypeId|numberDoctors|num_procedures|num_equipment|
+---------------------------------------+---------------------+--------------+-------------+--------------+-------------+
|Behind the Chief's Palace Abesim, Ghana|null                 |null          |NULL         |6             |0            |
|A & A Medlove Medical Centre           |null                 |null          |NULL         |10            |0            |
|Accra Specialist Eye Hospital          |null               

## 2D. Medical Desert Identification

In [0]:
desert_analysis = df_enriched.groupBy("address_stateOrRegion").agg(
    F.count("*").alias("facility_count"),
    F.sum(F.when(F.col("numberDoctors").isNotNull(), F.col("numberDoctors")).otherwise(0)).alias("total_doctors"),
    F.sum(F.when(F.col("capacity").isNotNull(), F.col("capacity")).otherwise(0)).alias("total_beds"),
    F.sum(F.when(F.col("facilityTypeId") == "hospital", 1).otherwise(0)).alias("hospital_count"),
    F.sum(F.when(F.lower(F.col("specialties")).contains("surgery"), 1).otherwise(0)).alias("has_surgery"),
    F.sum(F.when(F.lower(F.col("specialties")).contains("emergency"), 1).otherwise(0)).alias("has_emergency"),
    F.sum(F.when(F.lower(F.col("specialties")).contains("obstetrics"), 1).otherwise(0)).alias("has_obstetrics"),
    F.sum(F.when(F.lower(F.col("specialties")).contains("pediatrics"), 1).otherwise(0)).alias("has_pediatrics"),
    F.sum(F.when(F.col("flag_sparse_record") == True, 1).otherwise(0)).alias("sparse_records"),
)

desert_analysis = desert_analysis.withColumn("desert_score",
    F.lit(100)
    - (F.col("facility_count") * 2)
    - (F.col("total_doctors") * 1)
    - (F.col("has_surgery") * 10)
    - (F.col("has_emergency") * 10)
    - (F.col("has_obstetrics") * 5)
)

print("=== MEDICAL DESERT ANALYSIS (higher score = more underserved) ===")
desert_analysis.orderBy("desert_score", ascending=False).show(20, truncate=False)

=== MEDICAL DESERT ANALYSIS (higher score = more underserved) ===
+----------------------+--------------+-------------+----------+--------------+-----------+-------------+--------------+--------------+--------------+------------+
|address_stateOrRegion |facility_count|total_doctors|total_beds|hospital_count|has_surgery|has_emergency|has_obstetrics|has_pediatrics|sparse_records|desert_score|
+----------------------+--------------+-------------+----------+--------------+-----------+-------------+--------------+--------------+--------------+------------+
|Accra East            |1             |0            |0         |0             |0          |0            |0             |1             |0             |98          |
|Asutifi South         |1             |0            |0         |0             |0          |0            |0             |0             |0             |98          |
|Sissala West District |1             |0            |0         |1             |0          |0            |0        

## 2E. Save Enriched Data to Unity Catalog

In [0]:
ENRICHED_TABLE = f"{TABLE_PREFIX}.facilities_enriched"
DESERT_TABLE = f"{TABLE_PREFIX}.regional_analysis"

df_enriched.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(ENRICHED_TABLE)
spark.sql(f"ALTER TABLE {ENRICHED_TABLE} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"Saved {df_enriched.count()} enriched facilities to {ENRICHED_TABLE}")

desert_analysis.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DESERT_TABLE)
print(f"Saved regional analysis to {DESERT_TABLE}")

Saved 797 enriched facilities to hackathon_vf.healthcare.facilities_enriched
Saved regional analysis to hackathon_vf.healthcare.regional_analysis


In [0]:
print("=" * 60)
print("DATA ANALYSIS COMPLETE")
print("=" * 60)
print(f"\nTotal clean facilities: {df_enriched.count()}")
print(f"\nAnomalies found:")
for flag_col in [c for c in df_enriched.columns if c.startswith("flag_")]:
    count = df_enriched.filter(F.col(flag_col) == True).count()
    print(f"  {flag_col}: {count}")
print(f"\nTop medical deserts:")
for row in desert_analysis.orderBy("desert_score", ascending=False).limit(5).collect():
    print(f"  {row['address_stateOrRegion']}: score={row['desert_score']}, facilities={row['facility_count']}, doctors={row['total_doctors']}")
print(f"\nTables: {ENRICHED_TABLE}, {DESERT_TABLE}")
print("Next: Run notebook 03_vector_store")

DATA ANALYSIS COMPLETE

Total clean facilities: 797

Anomalies found:
  flag_procedures_no_doctors: 55
  flag_capacity_no_equipment: 10
  flag_clinic_claims_surgery: 19
  flag_too_many_specialties: 94
  flag_sparse_record: 0

Top medical deserts:
  Central Tongu District: score=98, facilities=1, doctors=0
  SH: score=98, facilities=1, doctors=0
  Ejisu Municipal: score=98, facilities=1, doctors=0
  Oti: score=98, facilities=1, doctors=0
  Ahafo: score=98, facilities=1, doctors=0

Tables: hackathon_vf.healthcare.facilities_enriched, hackathon_vf.healthcare.regional_analysis
Next: Run notebook 03_vector_store
